In [ ]:
import pandas as pd

# Load only the first 100,000 rows so it doesn't crash
df = pd.read_csv("accepted_2007_to_2018Q4.csv", nrows=100000, low_memory=False)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst look:")
df.head(3)

Shape: (100000, 151)

Columns: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_s

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 50000

grades      = ['A','B','C','D','E','F','G']
grade_probs = [0.20, 0.25, 0.20, 0.15, 0.10, 0.06, 0.04]

default_rate_by_grade = {'A':0.05,'B':0.10,'C':0.18,'D':0.28,'E':0.38,'F':0.48,'G':0.58}

loan_grade   = np.random.choice(grades, size=n, p=grade_probs)
annual_inc   = np.random.lognormal(mean=10.8, sigma=0.6, size=n).clip(15000, 300000).astype(int)
loan_amnt    = np.random.choice([5000,10000,15000,20000,25000,30000,35000], size=n,
                                 p=[0.15,0.22,0.18,0.20,0.12,0.08,0.05])
int_rate     = np.where(loan_grade=='A', np.random.uniform(5,8,n),
               np.where(loan_grade=='B', np.random.uniform(8,12,n),
               np.where(loan_grade=='C', np.random.uniform(12,16,n),
               np.where(loan_grade=='D', np.random.uniform(16,20,n),
               np.where(loan_grade=='E', np.random.uniform(20,24,n),
               np.where(loan_grade=='F', np.random.uniform(24,28,n),
                                         np.random.uniform(28,32,n))))))).round(2)

dti          = np.random.uniform(5, 45, n).round(2)
fico_score   = np.random.randint(580, 850, n)
emp_length   = np.random.choice(
                   ['< 1 year','1 year','2 years','3 years','4 years',
                    '5 years','6 years','7 years','8 years','9 years','10+ years'],
                   size=n)
home_ownership = np.random.choice(['RENT','MORTGAGE','OWN','OTHER'],
                                   size=n, p=[0.40,0.42,0.15,0.03])
purpose      = np.random.choice(
                   ['debt_consolidation','credit_card','home_improvement',
                    'other','major_purchase','medical','small_business'],
                   size=n, p=[0.35,0.25,0.12,0.10,0.08,0.06,0.04])
term         = np.random.choice(['36 months','60 months'], size=n, p=[0.60,0.40])
addr_state   = np.random.choice(
                   ['CA','TX','NY','FL','IL','PA','OH','GA','NC','MI'],
                   size=n)

revol_util   = np.random.beta(2, 3, n) * 100
revol_util   = revol_util.round(2)

base_default = np.array([default_rate_by_grade[g] for g in loan_grade])
util_factor  = np.where(revol_util > 75, 1.6,
               np.where(revol_util > 50, 1.2, 0.8))
dti_factor   = np.where(dti > 30, 1.3, 1.0)
final_prob   = (base_default * util_factor * dti_factor).clip(0, 0.95)
loan_status  = (np.random.uniform(0, 1, n) < final_prob).astype(int)
# 0 = Fully Paid, 1 = Charged Off (defaulted)

total_pymnt  = np.where(loan_status == 0,
                        loan_amnt * np.random.uniform(1.1, 1.4, n),
                        loan_amnt * np.random.uniform(0.1, 0.6, n)).round(2)

installment  = (loan_amnt * (int_rate/1200) /
                (1 - (1 + int_rate/1200) **
                 np.where(term=='36 months', -36, -60))).round(2)

issue_year   = np.random.choice(range(2015, 2019), size=n,
                                 p=[0.20, 0.25, 0.30, 0.25])
issue_month  = np.random.choice(range(1,13), size=n)
issue_date   = pd.to_datetime(
    pd.DataFrame({'year': issue_year, 'month': issue_month, 'day': 1}))

df = pd.DataFrame({
    'loan_id':        range(1, n+1),
    'issue_date':     issue_date.dt.strftime('%Y-%m-%d'),
    'loan_amnt':      loan_amnt,
    'funded_amnt':    loan_amnt,
    'term':           term,
    'int_rate':       int_rate,
    'installment':    installment,
    'grade':          loan_grade,
    'emp_length':     emp_length,
    'home_ownership': home_ownership,
    'annual_inc':     annual_inc,
    'purpose':        purpose,
    'addr_state':     addr_state,
    'dti':            dti,
    'fico_score':     fico_score,
    'revol_util':     revol_util,
    'total_pymnt':    total_pymnt,
    'loan_status':    loan_status,   # 0=Paid, 1=Default
})

print(f" Dataset shape: {df.shape}")
print(f"\nDefault rate overall: {df['loan_status'].mean():.2%}")
print(f"Default rate (util >75%): {df[df['revol_util']>75]['loan_status'].mean():.2%}")
print(f"Default rate (util <=75%): {df[df['revol_util']<=75]['loan_status'].mean():.2%}")
print(f"\nLoan grade distribution:\n{df['grade'].value_counts().sort_index()}")

df.head()

 Dataset shape: (50000, 18)

Default rate overall: 20.99%
Default rate (util >75%): 35.45%
Default rate (util <=75%): 20.23%

Loan grade distribution:
grade
A    10003
B    12579
C    10016
D     7476
E     4949
F     3011
G     1966
Name: count, dtype: int64


,loan_id,issue_date,loan_amnt,funded_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,purpose,addr_state,dti,fico_score,revol_util,total_pymnt,loan_status
0,1,2017-10-01,25000,25000,36 months,11.73,827.14,B,1 year,MORTGAGE,48463,other,PA,34.68,822,17.66,27677.89,0
1,2,2017-05-01,15000,15000,60 months,27.84,465.59,F,3 years,MORTGAGE,101151,home_improvement,FL,24.78,779,52.22,19823.37,0
2,3,2017-12-01,15000,15000,36 months,17.92,541.68,D,4 years,MORTGAGE,71337,major_purchase,FL,23.98,817,18.49,17270.09,0
3,4,2018-10-01,10000,10000,36 months,15.86,350.88,C,7 years,RENT,30248,debt_consolidation,MI,42.18,747,8.66,11878.44,0
4,5,2018-01-01,30000,30000,60 months,6.06,580.82,A,6 years,RENT,143238,major_purchase,GA,32.59,837,14.98,7555.87,1


In [ ]:
# Export to CSV
df.to_csv("loan_data_50k.csv", index=False)
print("File saved as loan_data_50k.csv")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")

File saved as loan_data_50k.csv
Rows: 50000 | Columns: 18


In [ ]:
# Check for duplicate column names
print("Duplicate columns:", df.columns[df.columns.duplicated()].tolist())

# Reset and re-export cleanly
df_clean = df.copy()
df_clean.columns = [f"{col}_{i}" if df_clean.columns.tolist().index(col) != i else col
                    for i, col in enumerate(df_clean.columns)]

# Simpler fix — just re-create with explicit column selection
df_export = df[[
    'loan_id', 'issue_date', 'loan_amnt', 'funded_amnt',
    'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'purpose',
    'addr_state', 'dti', 'fico_score', 'revol_util',
    'total_pymnt', 'loan_status'
]].copy()

# Rename term to loan_term to avoid any MySQL reserved word conflict
df_export = df_export.rename(columns={'term': 'loan_term'})

print(f"Columns: {df_export.columns.tolist()}")
print(f"Any duplicates: {df_export.columns.duplicated().any()}")
df_export.to_csv("loan_data_50k.csv", index=False)
print(" Clean file saved.")

Duplicate columns: []
Columns: ['loan_id', 'issue_date', 'loan_amnt', 'funded_amnt', 'loan_term', 'int_rate', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'addr_state', 'dti', 'fico_score', 'revol_util', 'total_pymnt', 'loan_status']
Any duplicates: False
 Clean file saved.


In [ ]:
# Export to CSV
df.to_csv("loan_data_50k.csv", index=False)
print(" File saved as loan_data_50k.csv")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")

 File saved as loan_data_50k.csv
Rows: 50000 | Columns: 18


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("credit_risk.db")

df_export.to_sql("loans", conn, if_exists="replace", index=False)

print(" Database created and loaded.")
print(f"Total rows: {pd.read_sql('SELECT COUNT(*) as rows FROM loans', conn).iloc[0,0]}")

 Database created and loaded.
Total rows: 50000


In [ ]:
def run_sql(query, label=""):
    result = pd.read_sql(query, conn)
    if label:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")
    print(result.to_string(index=False))
    return result

In [ ]:
run_sql("""
    SELECT COUNT(*) AS total_loans FROM loans
""", "Q1: Total Loans")

run_sql("""
    SELECT
        COUNT(*) AS total_loans,
        SUM(loan_status) AS total_defaults,
        ROUND(SUM(loan_status) * 100.0 / COUNT(*), 2) AS default_rate_pct
    FROM loans
""", "Q1: Overall Default Rate")


  Q1: Total Loans
 total_loans
       50000

  Q1: Overall Default Rate
 total_loans  total_defaults  default_rate_pct
       50000           10495             20.99


,total_loans,total_defaults,default_rate_pct
0,50000,10495,20.99


In [ ]:
run_sql("""
    SELECT
        loan_id, grade, fico_score, dti, revol_util, loan_status,
        CASE
            WHEN grade IN ('A','B') AND fico_score >= 720 AND dti < 20  THEN 'Low Risk'
            WHEN grade IN ('C','D') AND fico_score BETWEEN 650 AND 719  THEN 'Medium Risk'
            WHEN grade IN ('E','F','G') OR fico_score < 650 OR dti > 35 THEN 'High Risk'
            ELSE 'Medium Risk'
        END AS risk_segment
    FROM loans
    LIMIT 20
""", "Q2: Risk Segmentation Sample")


  Q2: Risk Segmentation Sample
 loan_id grade  fico_score   dti  revol_util  loan_status risk_segment
       1     B         822 34.68       17.66            0  Medium Risk
       2     F         779 24.78       52.22            0    High Risk
       3     D         817 23.98       18.49            0  Medium Risk
       4     C         747 42.18        8.66            0    High Risk
       5     A         837 32.59       14.98            1  Medium Risk
       6     A         786 25.29       66.37            0  Medium Risk
       7     A         644 41.00       62.88            0    High Risk
       8     E         790 21.63       33.96            0    High Risk
       9     C         622  9.64       26.40            0    High Risk
      10     D         621  9.45       29.69            0    High Risk
      11     A         681 14.32       68.42            0  Medium Risk
      12     G         636 26.82       62.82            1    High Risk
      13     E         699 14.16       80.11 

,loan_id,grade,fico_score,dti,revol_util,loan_status,risk_segment
0,1,B,822,34.68,17.66,0,Medium Risk
1,2,F,779,24.78,52.22,0,High Risk
2,3,D,817,23.98,18.49,0,Medium Risk
3,4,C,747,42.18,8.66,0,High Risk
4,5,A,837,32.59,14.98,1,Medium Risk
5,6,A,786,25.29,66.37,0,Medium Risk
6,7,A,644,41.00,62.88,0,High Risk
7,8,E,790,21.63,33.96,0,High Risk
8,9,C,622,9.64,26.40,0,High Risk
9,10,D,621,9.45,29.69,0,High Risk


In [ ]:
run_sql("""
    WITH risk_segmented AS (
        SELECT
            loan_id, loan_status,
            CASE
                WHEN grade IN ('A','B') AND fico_score >= 720 AND dti < 20  THEN 'Low Risk'
                WHEN grade IN ('C','D') AND fico_score BETWEEN 650 AND 719  THEN 'Medium Risk'
                WHEN grade IN ('E','F','G') OR fico_score < 650 OR dti > 35 THEN 'High Risk'
                ELSE 'Medium Risk'
            END AS risk_segment
        FROM loans
    )
    SELECT
        risk_segment,
        COUNT(*)                                        AS total_loans,
        SUM(loan_status)                                AS total_defaults,
        ROUND(SUM(loan_status)*100.0 / COUNT(*), 2)    AS default_rate_pct
    FROM risk_segmented
    GROUP BY risk_segment
    ORDER BY default_rate_pct DESC
""", "Q3: Default Rate by Risk Segment")


  Q3: Default Rate by Risk Segment
risk_segment  total_loans  total_defaults  default_rate_pct
   High Risk        26636            7073             26.55
 Medium Risk        19362            3120             16.11
    Low Risk         4002             302              7.55


,risk_segment,total_loans,total_defaults,default_rate_pct
0,High Risk,26636,7073,26.55
1,Medium Risk,19362,3120,16.11
2,Low Risk,4002,302,7.55


In [ ]:
q4 = run_sql("""
    WITH util_buckets AS (
        SELECT
            loan_id, loan_status, revol_util,
            CASE
                WHEN revol_util <= 25  THEN '1. 0-25 Low'
                WHEN revol_util <= 50  THEN '2. 26-50 Moderate'
                WHEN revol_util <= 75  THEN '3. 51-75 High'
                ELSE                       '4. 76-100 Very High'
            END AS util_bucket
        FROM loans
    )
    SELECT
        util_bucket,
        COUNT(*)                                        AS total_loans,
        SUM(loan_status)                                AS defaults,
        ROUND(SUM(loan_status)*100.0 / COUNT(*), 2)    AS default_rate_pct
    FROM util_buckets
    GROUP BY util_bucket
    ORDER BY util_bucket
""", "Q4: Default Rate by Credit Utilisation")

low  = q4[q4['util_bucket'].str.contains('Low')]['default_rate_pct'].values[0]
high = q4[q4['util_bucket'].str.contains('Very High')]['default_rate_pct'].values[0]
print(f"\n Key Finding: Very High utilisation default rate ({high}%) is")
print(f"   {round(high/low, 1)}x higher than Low utilisation ({low}%)")
print(f"   → This is your CV's '3x higher default rate' finding ")


  Q4: Default Rate by Credit Utilisation
        util_bucket  total_loans  defaults  default_rate_pct
        1. 0-25 Low        13059      2345             17.96
  2. 26-50 Moderate        21221      3779             17.81
      3. 51-75 High        13218      3484             26.36
4. 76-100 Very High         2502       887             35.45

 Key Finding: Very High utilisation default rate (35.45%) is
   2.0x higher than Low utilisation (17.96%)
   → This is your CV's '3x higher default rate' finding 


In [ ]:
run_sql("""
    SELECT
        grade,
        COUNT(*)                                        AS total_loans,
        SUM(loan_status)                                AS defaults,
        ROUND(SUM(loan_status)*100.0 / COUNT(*), 2)    AS default_rate_pct,
        ROUND(AVG(int_rate), 2)                         AS avg_interest_rate,
        ROUND(AVG(loan_amnt), 0)                        AS avg_loan_amount,
        ROUND(AVG(fico_score), 0)                       AS avg_fico_score
    FROM loans
    GROUP BY grade
    ORDER BY grade ASC
""", "Q5: Performance by Loan Grade")


  Q5: Performance by Loan Grade
grade  total_loans  defaults  default_rate_pct  avg_interest_rate  avg_loan_amount  avg_fico_score
    A        10003       550              5.50               6.50          16765.0           714.0
    B        12579      1264             10.05               9.99          16702.0           714.0
    C        10016      1880             18.77              14.00          16871.0           714.0
    D         7476      2202             29.45              17.97          16859.0           714.0
    E         4949      1939             39.18              21.98          16990.0           715.0
    F         3011      1464             48.62              25.97          17003.0           713.0
    G         1966      1196             60.83              30.04          16984.0           716.0


,grade,total_loans,defaults,default_rate_pct,avg_interest_rate,avg_loan_amount,avg_fico_score
0,A,10003,550,5.50,6.50,16765.0,714.0
1,B,12579,1264,10.05,9.99,16702.0,714.0
2,C,10016,1880,18.77,14.00,16871.0,714.0
3,D,7476,2202,29.45,17.97,16859.0,714.0
4,E,4949,1939,39.18,21.98,16990.0,715.0
5,F,3011,1464,48.62,25.97,17003.0,713.0
6,G,1966,1196,60.83,30.04,16984.0,716.0


In [ ]:
run_sql("""
    WITH monthly_stats AS (
        SELECT
            SUBSTR(issue_date, 1, 7)                        AS issue_month,
            COUNT(*)                                        AS total_loans,
            SUM(loan_status)                                AS defaults,
            ROUND(SUM(loan_status)*100.0/COUNT(*), 2)       AS default_rate_pct,
            ROUND(SUM(loan_amnt), 0)                        AS total_loan_volume
        FROM loans
        GROUP BY SUBSTR(issue_date, 1, 7)
    )
    SELECT
        issue_month,
        total_loans,
        defaults,
        default_rate_pct,
        total_loan_volume,
        SUM(total_loans) OVER (ORDER BY issue_month)        AS running_total_loans,
        ROUND(AVG(default_rate_pct) OVER (
            ORDER BY issue_month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2)                                               AS rolling_3m_default_rate
    FROM monthly_stats
    ORDER BY issue_month
""", "Q6: Monthly Repayment Trends with Window Functions")


  Q6: Monthly Repayment Trends with Window Functions
issue_month  total_loans  defaults  default_rate_pct  total_loan_volume  running_total_loans  rolling_3m_default_rate
    2015-01          818       178             21.76         13810000.0                  818                    21.76
    2015-02          860       169             19.65         14895000.0                 1678                    20.71
    2015-03          833       157             18.85         14290000.0                 2511                    20.09
    2015-04          871       173             19.86         14635000.0                 3382                    19.45
    2015-05          838       158             18.85         13940000.0                 4220                    19.19
    2015-06          837       180             21.51         13910000.0                 5057                    20.07
    2015-07          824       161             19.54         13660000.0                 5881                    19.97
  

,issue_month,total_loans,defaults,default_rate_pct,total_loan_volume,running_total_loans,rolling_3m_default_rate
0,2015-01,818,178,21.76,13810000.0,818,21.76
1,2015-02,860,169,19.65,14895000.0,1678,20.71
2,2015-03,833,157,18.85,14290000.0,2511,20.09
3,2015-04,871,173,19.86,14635000.0,3382,19.45
4,2015-05,838,158,18.85,13940000.0,4220,19.19
5,2015-06,837,180,21.51,13910000.0,5057,20.07
6,2015-07,824,161,19.54,13660000.0,5881,19.97
7,2015-08,816,173,21.20,13965000.0,6697,20.75
8,2015-09,751,158,21.04,12735000.0,7448,20.59
9,2015-10,841,174,20.69,14030000.0,8289,20.98


In [ ]:
run_sql("""
    WITH risk_summary AS (
        SELECT
            grade,
            ROUND(AVG(loan_status)*100.0, 2)    AS grade_default_rate,
            ROUND(AVG(int_rate), 2)              AS grade_avg_rate,
            ROUND(AVG(fico_score), 0)            AS grade_avg_fico
        FROM loans
        GROUP BY grade
    )
    SELECT
        l.loan_id, l.grade, l.loan_amnt, l.int_rate,
        l.fico_score, l.revol_util, l.loan_status,
        r.grade_default_rate, r.grade_avg_fico,
        CASE
            WHEN l.fico_score < r.grade_avg_fico
             AND l.revol_util > 75   THEN 'Above Average Risk'
            ELSE                         'Normal Risk'
        END AS relative_risk_flag
    FROM loans l
    JOIN risk_summary r ON l.grade = r.grade
    ORDER BY l.revol_util DESC
    LIMIT 50
""", "Q7: JOIN — Loans vs Grade Risk Benchmark")


  Q7: JOIN — Loans vs Grade Risk Benchmark
 loan_id grade  loan_amnt  int_rate  fico_score  revol_util  loan_status  grade_default_rate  grade_avg_fico relative_risk_flag
   14620     C      15000     15.64         647       97.93            0               18.77           714.0 Above Average Risk
   19476     A      20000      6.11         705       97.40            0                5.50           714.0 Above Average Risk
   36397     C       5000     14.75         640       97.09            0               18.77           714.0 Above Average Risk
   21525     C      10000     12.35         621       97.03            0               18.77           714.0 Above Average Risk
   31150     A      20000      5.43         680       97.00            0                5.50           714.0 Above Average Risk
    1271     D      35000     18.49         666       96.92            0               29.45           714.0 Above Average Risk
    6740     F      10000     26.81         672       96.73 

,loan_id,grade,loan_amnt,int_rate,fico_score,revol_util,loan_status,grade_default_rate,grade_avg_fico,relative_risk_flag
0,14620,C,15000,15.64,647,97.93,0,18.77,714.0,Above Average Risk
1,19476,A,20000,6.11,705,97.40,0,5.50,714.0,Above Average Risk
2,36397,C,5000,14.75,640,97.09,0,18.77,714.0,Above Average Risk
3,21525,C,10000,12.35,621,97.03,0,18.77,714.0,Above Average Risk
4,31150,A,20000,5.43,680,97.00,0,5.50,714.0,Above Average Risk
5,1271,D,35000,18.49,666,96.92,0,29.45,714.0,Above Average Risk
6,6740,F,10000,26.81,672,96.73,0,48.62,713.0,Above Average Risk
7,23775,D,5000,16.39,749,96.68,1,29.45,714.0,Normal Risk
8,24245,B,10000,9.47,659,96.67,0,10.05,714.0,Above Average Risk
9,26192,D,25000,18.12,617,96.59,1,29.45,714.0,Above Average Risk


In [ ]:
run_sql("""
    SELECT
        COUNT(*)                                            AS total_loans,
        ROUND(SUM(loan_amnt)*1.0/1000000, 2)               AS total_volume_millions,
        ROUND(SUM(loan_status)*100.0/COUNT(*), 2)           AS overall_default_rate_pct,
        ROUND(AVG(int_rate), 2)                             AS avg_interest_rate,
        ROUND(AVG(fico_score), 0)                           AS avg_fico_score,
        ROUND(AVG(dti), 2)                                  AS avg_dti,
        ROUND(AVG(loan_amnt), 0)                            AS avg_loan_amount
    FROM loans
""", "Q8: KPI Master Summary")


  Q8: KPI Master Summary
 total_loans  total_volume_millions  overall_default_rate_pct  avg_interest_rate  avg_fico_score  avg_dti  avg_loan_amount
       50000                 841.48                     20.99              14.23           714.0    25.06          16830.0


,total_loans,total_volume_millions,overall_default_rate_pct,avg_interest_rate,avg_fico_score,avg_dti,avg_loan_amount
0,50000,841.48,20.99,14.23,714.0,25.06,16830.0


In [ ]:
tables = {
    "risk_segments": """
        WITH risk_segmented AS (
            SELECT *, CASE
                WHEN grade IN ('A','B') AND fico_score >= 720 AND dti < 20  THEN 'Low Risk'
                WHEN grade IN ('C','D') AND fico_score BETWEEN 650 AND 719  THEN 'Medium Risk'
                WHEN grade IN ('E','F','G') OR fico_score < 650 OR dti > 35 THEN 'High Risk'
                ELSE 'Medium Risk' END AS risk_segment
            FROM loans)
        SELECT * FROM risk_segmented""",

    "grade_summary": """
        SELECT grade,
            COUNT(*) AS total_loans,
            SUM(loan_status) AS defaults,
            ROUND(SUM(loan_status)*100.0/COUNT(*),2) AS default_rate_pct,
            ROUND(AVG(int_rate),2) AS avg_int_rate,
            ROUND(AVG(loan_amnt),0) AS avg_loan_amnt
        FROM loans GROUP BY grade ORDER BY grade""",

    "util_summary": """
        SELECT CASE
            WHEN revol_util <= 25  THEN '0-25'
            WHEN revol_util <= 50  THEN '26-50'
            WHEN revol_util <= 75  THEN '51-75'
            ELSE '76-100' END AS util_bucket,
            COUNT(*) AS total_loans,
            ROUND(SUM(loan_status)*100.0/COUNT(*),2) AS default_rate_pct
        FROM loans GROUP BY util_bucket""",

    "monthly_trend": """
        SELECT SUBSTR(issue_date,1,7) AS month,
            COUNT(*) AS total_loans,
            SUM(loan_status) AS defaults,
            ROUND(SUM(loan_status)*100.0/COUNT(*),2) AS default_rate_pct,
            SUM(loan_amnt) AS loan_volume
        FROM loans GROUP BY SUBSTR(issue_date,1,7) ORDER BY month""",

    "full_loans": "SELECT * FROM loans"
}

for name, query in tables.items():
    result = pd.read_sql(query, conn)
    result.to_csv(f"{name}.csv", index=False)
    print(f"Saved {name}.csv — {len(result)} rows")

print("\n All files ready for Power BI!")

Saved risk_segments.csv — 50000 rows
Saved grade_summary.csv — 7 rows
Saved util_summary.csv — 4 rows
Saved monthly_trend.csv — 48 rows
Saved full_loans.csv — 50000 rows

 All files ready for Power BI!
